<a href="https://colab.research.google.com/github/covinolab/sbi_tutorial_algosb2026/blob/main/01_sbi_toy_2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SBI from scratch on a 2D toy problem

**AlgoSB 2026 · simulation-based inference · hands-on notebook 1 of 2 · 75 minutes**

This morning you heard how simulation-based inference (SBI) turns a simulator into a posterior. Now you do it yourself, on a problem small enough that the exact posterior is known. That exact posterior is your **answer key**: whatever you build, you can compare with it.

| Part | Min | Your challenge |
| --- | --- | --- |
| 0 | 5 | Setup and the three ingredients |
| 1 | 12 | Inference by hand, then by a loop |
| 2 | 18 | Train a posterior that matches the answer key |
| 3 | 10 | Find the hidden walls of a black-box network |
| 4 | 12 | Which network lies? |
| 5 | 13 | Pin θ down with 100 observations |
| 6 | 5 | Wrap-up |

**How it works**

- Every part is a challenge. A check cell scores what you built and prints ✅ PASS or ❌ NOT YET.
- How you get there is up to you: trial and error, changing the given code, thinking, or asking an LLM for code. The check is what counts.
- Every `sbi` call you need appears once in a worked cell. Copy it from there.
- After a pass, write one sentence on why it worked. We discuss each part in the room.
- Stuck? Ask us. Worked answers with the reasoning are in `solutions/`.

In [ ]:
# Setup. On Colab this installs sbi and fetches the tutorial files; on your own machine it does nothing.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("tutorial_utils.py"):
    !pip install -q sbi==0.27.0
    !git clone -q --depth 1 https://github.com/covinolab/sbi_tutorial_algosb2026.git
    %cd sbi_tutorial_algosb2026

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

import sbi
from sbi.inference import NLE, NPE
from sbi.neural_nets import posterior_nn
from sbi.utils import BoxUniform

import tutorial_utils as tu
from tutorial_utils import (ExactPosterior, check_by_hand, check_calibration, check_iid,
                            check_match, check_walls, interval_width, plot_cloud, plot_posteriors)

warnings.filterwarnings("ignore")  # sbi is talkative; none of its warnings matter today
tu.set_plot_style()
torch.manual_seed(0)
np.random.seed(0)
print(f"sbi {sbi.__version__} · torch {torch.__version__}")

## Part 0 · Setup and the three ingredients (5 min)

SBI needs three things from you.

1. A **prior** $p(\theta)$: what you know about the parameters before you see data.
2. A **simulator**: a program that turns parameters $\theta$ into data $x$. Running it draws $x \sim p(x \mid \theta)$. You never have to write down or evaluate that likelihood.
3. An **observation** $x_o$: the data you want explained.

From these you want the **posterior** $p(\theta \mid x_o)$: the parameters that could have produced $x_o$, each weighted by how plausible it is.

Both notebooks follow the same five steps with the same names:

| Step | In the code |
| --- | --- |
| 1. Prior | `prior` |
| 2. Simulator | `simulator(theta)` returns `x` |
| 3. Training set | `theta`, `x` |
| 4. Training | `posterior` |
| 5. Inference and checks | `x_o`, `samples` |

In [ ]:
# 1. Prior: each of the two parameters lies somewhere between -5 and 5
prior = BoxUniform(low=-5 * torch.ones(2), high=5 * torch.ones(2))

# 2. Simulator: one noisy 2D point around theta
def simulator(theta):
    return theta + 0.5 * torch.randn_like(theta)

# The observation: one 2D point. Somebody ran the simulator once, at a theta you are not told.
x_o = tu.load_toy_observations()[0]

LIMITS = [[-5, 5], [-5, 5]]
print("x_o =", x_o)

In [ ]:
# A first look: draw five parameters from the prior and simulate a data point for each
theta_demo = prior.sample((5,))
x_demo = simulator(theta_demo)
for t, x_ in zip(theta_demo, x_demo):
    print(f"theta = ({t[0]:5.2f}, {t[1]:5.2f})   ->   x = ({x_[0]:5.2f}, {x_[1]:5.2f})")

**Warm-up, no check.** Name the three ingredients in the cells above: which line is the prior, which the simulator, which the observation? What would each of the three be in your own research?

## Part 1 · Inference by hand, then by a loop (12 min)

The simulator runs forward: from θ to x. Inference is the way back: which θ could have produced $x_o$? Do it by hand first.

**Challenge 1a: the best θ.** Pick a θ, simulate a cloud of points, compare the cloud with $x_o$, adjust. Find a θ whose cloud lands on $x_o$.

**Challenge 1b: the furthest θ.** Now move away from your best θ. Find a second θ, as far from the first as you can, that could still have produced $x_o$.

**Passes when** the centre of the best cloud lies within 0.1 of $x_o$, and the furthest θ is at least 1.0 away from the best and still inside the 95% region of the answer key.

In [ ]:
theta_best = torch.tensor([0.0, 0.0])      # ← change me until the cloud lands on x_o
plot_cloud(theta_best, x_o, simulator);

In [ ]:
theta_far = torch.tensor([0.0, 0.0])       # ← as far from theta_best as x_o still allows
plot_cloud([theta_best, theta_far], x_o, simulator, zoom=3);

In [ ]:
check_by_hand(theta_best, theta_far, x_o)

**✍️ One sentence.** Why is there more than one θ that could have produced $x_o$? *(double-click to write)*

### The same search, done properly: a loop

You tried parameters, simulated, compared with $x_o$, and kept what fit. A loop can do the trying:

1. draw θ from the prior,
2. run the simulator once,
3. measure the distance between the simulated point and $x_o$,
4. keep θ if the distance is below a tolerance ε.

The kept parameters are samples from an approximate posterior. This is **rejection ABC** (approximate Bayesian computation). It needs no likelihood, only a simulator and a distance. This cell is a worked add-on without a check: run it and read it.

In [ ]:
epsilon = 0.3

kept = []
for _ in range(100_000):
    theta = prior.sample()              # propose parameters
    x = simulator(theta)                # simulate one data point
    if torch.dist(x, x_o) < epsilon:    # distance to the observation
        kept.append(theta)              # close enough: keep
kept = torch.stack(kept)

print(f"kept {len(kept)} of 100,000 simulations ({len(kept) / 1000:.2f}%)")

In [ ]:
# The kept parameters next to the answer key. Now it can be told: x_o was simulated at theta = (3, -1.5).
theta_true = torch.tensor([3.0, -1.5])
exact = ExactPosterior(x_o)                        # the answer key for x_o

axes = plot_posteriors([kept, exact.sample((2000,))], ["rejection ABC", "answer key"],
                       truth=theta_true, limits=[[0.5, 4.5], [-3.2, 0.8]])
axes[0].scatter(*theta_best, marker="P", s=110, color="black", label="your best θ")
axes[0].scatter(*theta_far, marker="X", s=110, color="black", label="your furthest θ")
axes[0].legend(loc="lower left", fontsize=8);

Your best θ sits in the middle of the kept cloud, your furthest θ at its rim. And the θ that $x_o$ was really simulated with is just one of the plausible parameters. It is not the best-fitting one.

The tolerance ε sets a trade-off: a large ε keeps parameters that are too far off, a small ε keeps almost nothing. Either way the loop throws most simulations away, and for a new observation it has to run again.

## Part 2 · Train a posterior that matches (18 min)

Rejection ABC used 0.3% of its simulations. **Neural posterior estimation (NPE)** uses all of them: a neural network $q(\theta \mid x)$ is trained on pairs $(\theta, x)$ to return the distribution of θ for a given x. No likelihood is evaluated, and no simulation is thrown away. Once trained, the network answers for any observation without new simulations.

The cell below runs steps 3 to 5 of the recipe. It works, but its settings are poor.

**Challenge.** Change the code until your posterior matches the answer key. Then go for the record: how few simulations are enough?

**Passes when** the overlay shows your posterior on top of the answer key and the **mismatch score** confirms it at 0.55 or below. The mismatch score is a classifier two-sample test (`c2st` in `sbi.utils.metrics`): a classifier is trained to tell your samples from samples of the answer key. A score of 0.5 means it cannot tell them apart, 1.0 means it always can.

In [ ]:
num_simulations = 50                                    # ← how few are enough?

# 3. Training set: draw parameters from the prior and simulate one data point for each
theta = prior.sample((num_simulations,))
x = simulator(theta)

# 4. Training: a neural network learns q(theta | x) from the pairs
density_estimator = posterior_nn(model="nsf", hidden_features=2, num_transforms=1)   # ← flexible enough?
inference = NPE(prior=prior, density_estimator=density_estimator)
inference.append_simulations(theta, x).train()
posterior = inference.build_posterior()

# 5. Inference: samples from q(theta | x_o)
samples = posterior.sample((2000,), x=x_o)

plot_posteriors([samples, exact.sample((2000,))], ["your NPE posterior", "answer key"], limits=LIMITS);

In [ ]:
check_match(samples, num_simulations, x_o)

**✍️ One sentence.** What did you change, and why did it work?

**🗣 On the board:** your lowest number of simulations that passes three runs in a row. With few simulations a single pass can be luck, so run the training cell and the check again with the same number. The check counts the passes in a row for you.

## Part 3 · Find the hidden walls (10 min)

Your network never saw $x_o$ during training, and still it answered. It answers for any observation. This is called **amortized** inference: you pay for the simulations and the training once, and every further observation is nearly free.

In [ ]:
# Your posterior at three other observations, each scored against its own answer key
for x_other in ([0.0, 0.0], [-3.0, 2.0], [2.0, -4.0]):
    x_other = torch.tensor(x_other)
    samples_other = posterior.sample((2000,), x=x_other, show_progress_bars=False)
    score = tu.mismatch_score(samples_other, ExactPosterior(x_other).sample((2000,)))
    print(f"x = {x_other.tolist()}: mismatch score {score:.2f}")

Inside its prior, your network is about as good everywhere as it is at $x_o$. But where does that end?

Here is a second network, handed over as a **black box**. Somebody else trained it for the same simulator, with a prior you are not told. All you can do is ask it for samples.

In [ ]:
black_box = tu.load_black_box()

probe = black_box.sample((2000,), x=x_o)
plot_posteriors([probe, exact.sample((2000,))], ["black box", "answer key"], limits=LIMITS);

At $x_o$ the black box agrees with the answer key. It looks trustworthy.

**Challenge.** Find out where the black box can be trusted: locate the four edges of its training prior.

**Passes when** all four edges are located to within 0.3.

Probe the black box, not your own network from Part 2. Far outside its prior a network may put all its samples outside the prior, and `sbi`, which throws those away and draws again, then never finishes. The black box has a time limit built in. Bonus 2 shows how to set one yourself.

In [ ]:
x_probe = torch.tensor([0.0, 0.0])                 # ← ask the black box about any observation you like
probe = black_box.sample((1000,), x=x_probe)
plot_posteriors(probe, "black box", limits=[[-8, 8], [-8, 8]]);

In [ ]:
wall_low = [-5.0, -5.0]       # ← your estimate: lower edge of θ₁, lower edge of θ₂
wall_high = [5.0, 5.0]        # ← your estimate: upper edge of θ₁, upper edge of θ₂

check_walls(wall_low, wall_high)

**✍️ One sentence.** What does the black box do with an observation its prior cannot explain, and why is that dangerous?

## Part 4 · Which network lies? (12 min)

Three more trained networks, A, B and C. All three were trained for this problem with 10,000 simulations. Each of them is either **overconfident** (its posteriors are too narrow), **underconfident** (too wide) or **fine**. You are not told how many of each kind there are.

In [ ]:
networks = tu.load_three_networks()      # {"A": network, "B": network, "C": network}

plot_posteriors([net.sample((2000,), x=x_o) for net in networks.values()],
                [f"network {name}" for name in networks], limits=[[0, 5], [-4, 1]]);

They disagree, and each of them looks like a reasonable posterior. Which width is the right one? In a real problem there is no answer key to settle it, so in this part **the answer key is off limits**. What you always have is the simulator.

The idea of a **coverage check**: if a posterior is right, the parameter that was used to simulate a data set falls inside the posterior's 90% interval nine times out of ten, and inside its 50% interval every second time. An overconfident posterior misses more often. An underconfident one misses less often.

The cell below shows how to get an interval from samples.

**Challenge.** Find out which network is which, with the simulator and a coverage check. Everything you need has appeared by now: `prior.sample()`, `simulator(theta)`, `network.sample((500,), x=...)`, and `torch.quantile`.

**Passes when** all three labels are right.

In [ ]:
# The 90% interval of θ₁ that network A reports for x_o: 5% of its samples lie below it, 5% above
samples_A = networks["A"].sample((2000,), x=x_o)
low, high = torch.quantile(samples_A[:, 0], torch.tensor([0.05, 0.95]))
print(f"network A says: with 90% probability θ₁ lies between {low:.2f} and {high:.2f}")

In [ ]:
# Your coverage check

In [ ]:
labels = {"A": "fine", "B": "fine", "C": "fine"}       # ← "overconfident", "underconfident" or "fine"

check_calibration(labels)

**✍️ One sentence.** Why can the simulator tell you that a posterior is wrong, without telling you what the right one is?

## Part 5 · Pin θ down (13 min)

One noisy point leaves θ uncertain by ±0.5. Now you get 100 points, all simulated independently with the same hidden θ. With more data the posterior has to get narrower.

In [ ]:
x_iid = tu.load_toy_observations()        # shape (100, 2); the first point is the x_o you know
print(x_iid.shape)

**The obstacle.** Your NPE network was trained on one point per θ. It cannot take five:

In [ ]:
try:
    posterior.sample((1000,), x=x_iid[:5])
except Exception as error:
    print("NPE says:", error)

separate = posterior.sample_batched((1000,), x=x_iid[:5], show_progress_bars=False)
print("\nsample_batched returns shape", tuple(separate.shape), "- five separate posteriors, one per point, not one for all five")

**The switch.** Neural *likelihood* estimation (NLE) learns the other conditional: $q(x \mid \theta)$, the likelihood of **one** observation. Independent observations multiply, so their log-likelihoods add:

$$\log p(\theta \mid x_1, \dots, x_n) = \log p(\theta) + \sum_{i=1}^{n} \log q(x_i \mid \theta) + \text{const.}$$

NLE trains on the same pairs $(\theta, x)$ as NPE. What it gives you is a function you can evaluate, not a sampler, so `sbi` draws the posterior samples with MCMC. This worked cell trains NLE on your training set from Part 2 and asks it about the first five observations.

In [ ]:
# 4. Training: the same pairs (theta, x) as in Part 2, a different conditional
inference = NLE(prior=prior, density_estimator="mdn")
inference.append_simulations(theta, x).train()
posterior_nle = inference.build_posterior(
    mcmc_method="slice_np_vectorized",
    mcmc_parameters=dict(num_chains=20, thin=1, warmup_steps=20),
)

# 5. Inference: the observation is now a stack of points, shape (n, 2)
n = 5
samples_nle = posterior_nle.sample((1000,), x=x_iid[:n])

plot_posteriors([samples_nle, ExactPosterior(x_iid[:n]).sample((1000,))],
                [f"NLE, {n} observations", "answer key"], truth=theta_true, limits=[[1, 5], [-3.5, 0.5]]);

`ExactPosterior(x_iid[:n])` is the answer key for the first n points, and `interval_width(samples)` returns the width of the 90% interval of each parameter.

**Challenge.** Produce one plot of how confident the posterior is against the number of observations: the width of its 90% interval for n from 1 to 100, with the answer key on top. Then read off your plot how many observations it takes to get the standard deviation of θ₁ down to 0.1.

**Passes when** the plot covers at least five values of n between 1 and 100, your widths stay within 25% of the answer key's, and your reading of n is right. For each n use the first n points, `x_iid[:n]`.

If your widths drift away from the answer key, improve the NLE training until they pass. (Short on time? `theta, x = tu.load_toy_training_set()` hands you 10,000 simulated pairs, and `posterior_nle = tu.load_toy_nle()` an NLE posterior trained on them.)

In [ ]:
# Your loop and your plot

In [ ]:
n_values = [1]        # ← the numbers of observations you used, at least five
widths = [1.0]        # ← width of the 90% interval of θ₁ for each of them
n_needed = 1          # ← observations needed for a standard deviation of 0.1

check_iid(n_values, widths, n_needed)

**✍️ One sentence.** What did the switch from NPE to NLE buy you, and what did it cost?

## Part 6 · Wrap-up (5 min)

**Say it in one sentence each:** when would you use NPE, and when NLE?

In [ ]:
tu.scorecard()

**What you did.** You ran the five steps of SBI end to end: prior, simulator, training set, training, inference. And you checked the result three ways: against the exact answer, against the limits of the training prior, and with a coverage test that needs no exact answer.

Notebook 2 repeats the same five steps on a physical problem where no answer key exists: a Brownian particle in a double well, observed as a time series.

---
## Bonus challenges

For those who finish early. Each has a check; none is needed for Notebook 2.

### Bonus 1 · Bend the simulator

Change one line of the simulator so that $x_1$ depends on $\theta_1^2$ instead of $\theta_1$. Train NPE on the new simulator and look at the posterior for $x_o$. What happened to it, and why?

**Passes when** your posterior puts between 40% and 60% of its samples on each side of $\theta_1 = 0$, with a gap in between: less than 5% of them within $|\theta_1| < 0.5$.

In [ ]:
# Your bent simulator, your training, your samples_bent



# tu.check_bimodal(samples_bent)

### Bonus 2 · Repair Part 3

The observation `x_far` below lies beyond the walls at ±5: with the prior of this notebook, no θ can explain it well. Retrain NPE with a prior wide enough that `x_far` gets a posterior that passes the Part 2 check.

**Passes when** the mismatch score at `x_far` is 0.55 or below, against the answer key for your new prior.

First, what your network from Part 2 says about `x_far`. Asked without a time limit, it may never answer: if none of its samples fall inside the prior, `sbi` keeps drawing. `max_sampling_time` stops that.

In [ ]:
x_far = torch.tensor([7.0, -6.5])

try:
    samples_old = posterior.sample((2000,), x=x_far, max_sampling_time=5.0, return_partial_on_timeout=True)
    print(f"after at most 5 seconds: {len(samples_old)} of 2000 samples inside the prior")
except RuntimeError:
    print("after 5 seconds: not a single sample inside the prior")

In [ ]:
# Your wider prior, your training, your samples_far



# tu.check_repair(samples_far, num_simulations_far, x_far, low=..., high=...)

### Bonus 3 · Part 5 with NPE after all

NPE can take several observations if you give it an *embedding network* that turns a set of points into a fixed-size summary, the same whatever the order of the points. Follow the sbi guide [How to use permutation-invariant embeddings for iid observations](https://sbi.readthedocs.io/en/stable/how_to_guide/08_permutation_invariant_embeddings.html) and reach the Part 5 benchmark with NPE instead of NLE.

**Passes when** `check_iid` passes with widths from your NPE posterior. It is enough to cover n from 1 to 25.

In [ ]:
# Your permutation-invariant NPE



# check_iid(n_values, widths_npe, n_needed, key="Bonus · NPE on sets")

### Bonus 4 · NLE against NRE

Neural ratio estimation (NRE) learns the ratio $p(x \mid \theta) / p(x)$ with a classifier and combines IID observations the same way as NLE. `NRE` is in `sbi.inference` and is used like `NLE`. Run it on the Part 5 benchmark and compare the two.

**Passes when** `check_iid` passes with widths from your NRE posterior.

In [ ]:
# Your NRE



# check_iid(n_values, widths_nre, n_needed, key="Bonus · NRE")